In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

In [2]:
def generate_temperature_data(seq_length, num_samples):
    np.random.seed(42)
    base_temp = 20      # Average temperature in degrees Celsius
    temp_variation = 5  # Maximum daily temperature variation

    data = base_temp + temp_variation * np.sin(np.linspace(0, 100, num_samples)) + np.random.normal(0, 1, num_samples)

    X = []
    y = []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])

    return np.array(X), np.array(y)

In [3]:
seq_length = 30
num_samples = 365  # Simulating a year of data
X, y = generate_temperature_data(seq_length, num_samples)
X = X[..., np.newaxis]  # Add an extra dimension for compatibility with RNN input

In [4]:
model = Sequential([
    SimpleRNN(50, activation='tanh', input_shape=(seq_length, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 50)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,651 (10.36 KB)

 Trainable params: 2,651 (10.36 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 324.2951 - val_loss: 323.8203
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 292.6019 - val_loss: 293.8474
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 264.4876 - val_loss: 265.4213
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 235.3884 - val_loss: 234.4447
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 205.9403 - val_loss: 205.7182
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 178.7389 - val_loss: 178.1613
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 153.6781 - val_loss: 154.3582
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 132.4460 - val_loss: 134.5059
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 114.9901 - val_loss: 117.1507
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 99.3373 - val_loss: 101.8485
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 85.7562 - val_loss: 88.3960
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms

In [6]:
def predict_future(model, recent_data, days):
    predictions = []
    input_seq = recent_data[-seq_length:]

    for _ in range(days):
        pred = model.predict(input_seq[np.newaxis, ...])[0, 0]
        predictions.append(pred)
        input_seq = np.append(input_seq[1:], [[pred]], axis=0)

    return predictions

In [7]:
future_predictions = predict_future(model, X[-1], 7)
print("Predicted Temperatures for Next 7 Days:", future_predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Predicted Temperatures for Next 7 Days: [np.float32(14.769442), np.float32(14.762617), np.float32(14.762371), np.float32(14.762371), np.float32(14.762371), np.float32(14.762371), np.float32(14.762371)]
